# How to Load Tabular Data

The fastest way to get music data into TimeToAlign! is through **tabular
loaders**. If your data is in CSV or TSV format, you're just 3 lines of
code away from analysis.

**What you'll learn:**
- Load music annotations from TSV/CSV files
- Access event counts, coordinate ranges, and metadata
- Create timelines from loaded data
- Write a custom loader that maps your columns to event fields
- Promote selected columns to typed fields with `column_specs`
- Reach nested JSON columns with `Field`

**Time:** 15 minutes

This is the gentle introduction. For the full column-to-field mechanism —
the Step-1 resolution chain, composite columns, and Step-2 field
promotion — see the in-depth CSV/TSV how-to.

## TL;DR

```python
from timetoalign.loader.tabular import TsvLoader

loader = TsvLoader()
loader.load("beethoven.notes.tsv")

df = loader.events.to_dataframe()    # Get as DataFrame
timeline = loader.create_timeline()  # Create Timeline
```

## Setup

In [1]:
from timetoalign.testdata import ensure_data

BEETHOVEN = ensure_data("score") / "beethoven_woo71"
THORESEN = ensure_data("thoresen")

# Available files
{
    "Beethoven files": [f.name for f in BEETHOVEN.glob("WoO71.*.tsv")],
    "Thoresen files": [f.name for f in THORESEN.glob("*.tsv")],
}

{'Beethoven files': ['WoO71.chords.tsv',
  'WoO71.measures.tsv',
  'WoO71.notes.tsv'],
 'Thoresen files': ['thoresen_test_h.tsv', 'thoresen_test.tsv']}

## Loading Notes from TSV

`TsvLoader` is the generic starting point for a TSV whose columns you
control. Declare the coordinate columns and promote the values you want as
typed fields. For a standard ms3 score export, use the score `Ms3Loader`
instead; it understands its notes, measures, chords, and harmonies facets.

The file gives each note's length twice: `duration_qb` in quarters, written
as a decimal (a twelfth as `0.08333333333333333`), and `duration` in whole
notes, written as an exact ratio (`1/48`). On an exact (`Fraction`) axis
the loader completes each end as start + duration exactly, and for a note
starting at `529/12` the sum with that decimal's binary value needs a
numerator wider than the 64 bits an exact value is stored in, so a
`duration_column = "duration_qb"` is refused rather than guessed back into
a ratio. Complete the ends from the exact `duration` column instead, with a
`ComputedField` as below, or declare `coordinate_type = NumberType.float`
for a file that gives its durations only as decimals.

In [2]:
from fractions import Fraction  # noqa: E402

import pyarrow as pa  # noqa: E402

from timetoalign.core import NumberType, RedundantNumberField, TimeUnit  # noqa: E402
from timetoalign.loader import ComputedField  # noqa: E402
from timetoalign.loader.tabular import TsvLoader  # noqa: E402


def end_in_quarters(table: pa.Table) -> pa.Array:
    """A note's end in quarters: its start plus four times its whole-note duration."""
    starts = table["quarterbeats"].to_pylist()
    wholes = table["duration"].to_pylist()
    # Written as "n/d" text, which the loader reads exactly.
    return pa.array(
        [
            None if s is None else str(Fraction(s) + 4 * Fraction(w))
            for s, w in zip(starts, wholes)
        ]
    )


class BeethovenNotesLoader(TsvLoader):
    """Generic TSV configuration for the coordinate and pitch columns used here."""

    start_column = "quarterbeats"
    end_column = ComputedField("end", expr=end_in_quarters)
    _default_unit = TimeUnit.quarters
    coordinate_type = NumberType.fraction
    # The whole-note column keeps its exact values under its own name, so it
    # is not read as the events' duration, which is in quarters.
    column_specs = {
        "duration": RedundantNumberField(name="duration_whole"),
        "midi": int,
        "staff": int,
        "voice": int,
    }


loader = BeethovenNotesLoader()
loader.load(BEETHOVEN / "WoO71.notes.tsv")

f"{len(loader.events):,} notes loaded"

'4,753 notes loaded'

## Converting to pandas

Use `to_dataframe()` to get a DataFrame with clean coordinate values:

In [3]:
loader.events.to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,duration_whole,midi,staff,...,timesig,gracenote,nominal_duration,scalar,tied,tpc,octave,chord_id,locator,extraction_id
0,event_2hl0c92gc6rs1,A3,interval,Event,0,1,1,1/4,57,2,...,2/4,NaN,1/4,1,NaN,3,3,3,row:0,extraction_1v0bc40f0b25n
1,event_382b86aywx9tn,E4,interval,Event,0,1,1,1/4,64,1,...,2/4,NaN,1/4,1,NaN,4,4,2,row:1,extraction_1v0bc40f0b25n
2,event_33jwk0hnadj2q,A4,interval,Event,0,1/2,1/2,1/8,69,1,...,2/4,NaN,1/8,1,NaN,3,4,0,row:2,extraction_1v0bc40f0b25n
3,event_1f9xmm403mh03,C#5,interval,Event,0,1/2,1/2,1/8,73,1,...,2/4,NaN,1/8,1,NaN,7,5,0,row:3,extraction_1v0bc40f0b25n
4,event_3ielvqoimn5qn,E5,interval,Event,1/2,1,1/2,1/8,76,1,...,2/4,NaN,1/8,1,NaN,4,5,1,row:4,extraction_1v0bc40f0b25n


## Quick Statistics

The loader provides immediate access to summary information:

In [4]:
{
    "event_count": len(loader.events),
    "coordinate_range": loader.events.coordinate_range(),
    "unit": str(loader.unit),
    "number_type": str(loader.number_type),
}

{'event_count': 4753,
 'coordinate_range': (Fraction(0, 1), Fraction(1753, 2)),
 'unit': 'quarters',
 'number_type': 'fraction'}

## Creating Timelines

TimeToAlign! represents temporal data as **Timelines**:

In [5]:
timeline = loader.create_timeline(uid="beethoven_notes")
timeline

ContinuousLogicalTimeline(id='beethoven_notes', length=1753/2, unit=quarters, events=4753, children=0)

## Custom Loaders for Non-Standard Formats

For files that don't match the ms3 format, create a custom loader by
subclassing `TsvLoader` or `CsvLoader`. You point the canonical column
attributes (`start_column`, `duration_column`, …) at the names in your
file.

Let's load the Thoresen annotations file, which has a different column
structure:

In [6]:
import pandas as pd  # noqa: E402

pd.read_csv(THORESEN / "thoresen_test.tsv", sep="\t", nrows=3)

,event_id,alignment_group_id,start_time_sec,duration_sec,event_type,graphical_element_id,image_filename,rect_coords_json,text_content,text_anchor_xy_json,layer_order,description
0,annot_cue_001,NaN,0.0,5.0,rectangle,rect_a,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,NaN,NaN,NaN
1,annot_cue_002,NaN,1.5,4.0,rectangle,rect_b,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,NaN,NaN,NaN
2,annot_cue_003,NaN,3.5,2.0,rectangle,rect_c,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,NaN,NaN,NaN


### The Simplest Custom Loader

Map the core coordinate columns and nothing else. Every source column
that you do *not* name survives as an opaque **property column** —
carried alongside the event so you never lose data, but left untyped:

In [7]:
from timetoalign.loader.tabular import TsvLoader  # noqa: E402


class ThoresenLoader(TsvLoader):
    """Minimal loader — maps the core event fields."""

    id_column = "event_id"
    start_column = "start_time_sec"
    duration_column = "duration_sec"
    event_type_column = "event_type"
    name_column = "description"

    _default_unit = TimeUnit.seconds
    coordinate_type = NumberType.float


thoresen = ThoresenLoader()
thoresen.load(THORESEN / "thoresen_test.tsv")
thoresen.events.to_dataframe()

,id,name,temporal_type,event_type,start,end,duration,alignment_group_id,graphical_element_id,image_filename,rect_coords_json,text_content,text_anchor_xy_json,layer_order,locator,extraction_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,NaN,rect_a,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,NaN,NaN,row:0 key:annot_cue_001,extraction_1y17ibu2cect2
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,NaN,rect_b,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,NaN,NaN,row:1 key:annot_cue_002,extraction_1y17ibu2cect2
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,NaN,rect_c,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,NaN,NaN,row:2 key:annot_cue_003,extraction_1y17ibu2cect2
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,NaN,rect_a2,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 145, ""y"": 90, ""width"": 160, ""height"": 58}",NaN,NaN,NaN,row:3 key:annot_cue_004,extraction_1y17ibu2cect2
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,NaN,rect_h2,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 385, ""y"": 46, ""width"": 139, ""height"": 20}",NaN,NaN,NaN,row:4 key:annot_cue_005,extraction_1y17ibu2cect2
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,NaN,rect_d3,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 310, ""y"": 93, ""width"": 154, ""height"": 18}",NaN,NaN,NaN,row:5 key:annot_cue_006,extraction_1y17ibu2cect2
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,NaN,rect_b3,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 456, ""y"": 69, ""width"": 229, ""height"": 18}",NaN,NaN,NaN,row:6 key:annot_cue_007,extraction_1y17ibu2cect2
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,NaN,rect_i4,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 14, ""y"": 115, ""width"": 127, ""height"": 31}",NaN,NaN,NaN,row:7 key:annot_cue_008,extraction_1y17ibu2cect2
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,NaN,rect_a4,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 663, ""y"": 82, ""width"": 97, ""height"": 23}",NaN,NaN,NaN,row:8 key:annot_cue_009,extraction_1y17ibu2cect2
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,NaN,rect_i5,thoresen_2010_form-building-patterns_p90-91_pa...,"{""x"": 19, ""y"": 119, ""width"": 251, ""height"": 29}",NaN,NaN,NaN,row:9 key:annot_cue_010,extraction_1y17ibu2cect2


### Promoting Columns to Typed Fields with `column_specs`

When a property column should become a typed field, name it in
`column_specs`. The keys are source-column names; the values are
anything the loader can resolve to a field — a bare Python type
(`int` / `float` / `str`) is the simplest form. Here we give two
extra columns explicit types:

In [8]:
class ThoresenTypedLoader(TsvLoader):
    """Selected columns promoted to typed fields."""

    id_column = "event_id"
    start_column = "start_time_sec"
    duration_column = "duration_sec"
    event_type_column = "event_type"
    name_column = "description"

    _default_unit = TimeUnit.seconds
    coordinate_type = NumberType.float

    # Promote these source columns to typed fields.
    column_specs = {
        "image_filename": str,
        "graphical_element_id": str,
    }


typed = ThoresenTypedLoader()
typed.load(THORESEN / "thoresen_test.tsv")
typed.events.to_dataframe()

,id,name,temporal_type,event_type,start,end,duration,image_filename,graphical_element_id,alignment_group_id,rect_coords_json,text_content,text_anchor_xy_json,layer_order,locator,extraction_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a,NaN,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,NaN,NaN,row:0 key:annot_cue_001,extraction_0q7ze135fhs95
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_b,NaN,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,NaN,NaN,row:1 key:annot_cue_002,extraction_0q7ze135fhs95
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_c,NaN,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,NaN,NaN,row:2 key:annot_cue_003,extraction_0q7ze135fhs95
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a2,NaN,"{""x"": 145, ""y"": 90, ""width"": 160, ""height"": 58}",NaN,NaN,NaN,row:3 key:annot_cue_004,extraction_0q7ze135fhs95
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_h2,NaN,"{""x"": 385, ""y"": 46, ""width"": 139, ""height"": 20}",NaN,NaN,NaN,row:4 key:annot_cue_005,extraction_0q7ze135fhs95
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,thoresen_2010_form-building-patterns_p90-91_pa...,rect_d3,NaN,"{""x"": 310, ""y"": 93, ""width"": 154, ""height"": 18}",NaN,NaN,NaN,row:5 key:annot_cue_006,extraction_0q7ze135fhs95
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_b3,NaN,"{""x"": 456, ""y"": 69, ""width"": 229, ""height"": 18}",NaN,NaN,NaN,row:6 key:annot_cue_007,extraction_0q7ze135fhs95
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_i4,NaN,"{""x"": 14, ""y"": 115, ""width"": 127, ""height"": 31}",NaN,NaN,NaN,row:7 key:annot_cue_008,extraction_0q7ze135fhs95
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a4,NaN,"{""x"": 663, ""y"": 82, ""width"": 97, ""height"": 23}",NaN,NaN,NaN,row:8 key:annot_cue_009,extraction_0q7ze135fhs95
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_i5,NaN,"{""x"": 19, ""y"": 119, ""width"": 251, ""height"": 29}",NaN,NaN,NaN,row:9 key:annot_cue_010,extraction_0q7ze135fhs95


That is the entire idea: **columns are a source artefact; fields are a
TimeToAlign! artefact.** `column_specs` is the bridge. A bare type is
the gentlest entry — the in-depth CSV/TSV how-to covers the full
resolution chain (composite columns, semantic pitch/id fields, and the
Step-2 `field_specs` promotion stage) for richer formats.

## Nested JSON Column Access with `Field`

The Thoresen data has a `rect_coords_json` column containing pixel
coordinates as JSON:
```json
{"x": 10, "y": 90, "width": 148, "height": 55}
```

Use `Field("column", "nested_field")` to point a coordinate attribute
straight at a nested value. TimeToAlign! parses the JSON
automatically. `ComputedField` lets you derive a coordinate from a
small formula over those nested values:

In [9]:
from timetoalign.loader import Field  # noqa: E402


class ThoresenGraphicalLoader(TsvLoader):
    """Loader using PIXEL coordinates from a nested JSON column."""

    # Nested fields are addressed directly; JSON is parsed automatically.
    start_column = Field("rect_coords_json", "x")
    end_column = ComputedField(
        "end", formula="rect_coords_json.x + rect_coords_json.width"
    )

    _default_unit = TimeUnit.pixels
    coordinate_type = NumberType.float
    default_event_type = "Rectangle"


graphical = ThoresenGraphicalLoader()
graphical.load(THORESEN / "thoresen_test.tsv")

{
    "unit": str(graphical.unit),
    "coordinate_range": graphical.events.coordinate_range(),
}

{'unit': 'pixels', 'coordinate_range': (10, 760)}

### Two Coordinate Systems from One File

The same TSV file backs timelines in different coordinate systems —
seconds from the time columns, pixels from the JSON column:

In [10]:
# Physical timeline (seconds)
typed.events.to_dataframe()

,id,name,temporal_type,event_type,start,end,duration,image_filename,graphical_element_id,alignment_group_id,rect_coords_json,text_content,text_anchor_xy_json,layer_order,locator,extraction_id
0,annot_cue_001,NaN,interval,rectangle,0.0,5.00,5.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a,NaN,"{""x"": 10, ""y"": 90, ""width"": 148, ""height"": 55}",NaN,NaN,NaN,row:0 key:annot_cue_001,extraction_0q7ze135fhs95
1,annot_cue_002,NaN,interval,rectangle,1.5,5.50,4.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_b,NaN,"{""x"": 40, ""y"": 37, ""width"": 127, ""height"": 21}",NaN,NaN,NaN,row:1 key:annot_cue_002,extraction_0q7ze135fhs95
2,annot_cue_003,NaN,interval,rectangle,3.5,5.50,2.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_c,NaN,"{""x"": 111, ""y"": 60, ""width"": 57, ""height"": 23}",NaN,NaN,NaN,row:2 key:annot_cue_003,extraction_0q7ze135fhs95
3,annot_cue_004,NaN,interval,rectangle,34.6,39.80,5.20,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a2,NaN,"{""x"": 145, ""y"": 90, ""width"": 160, ""height"": 58}",NaN,NaN,NaN,row:3 key:annot_cue_004,extraction_0q7ze135fhs95
4,annot_cue_005,NaN,interval,rectangle,43.5,48.00,4.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_h2,NaN,"{""x"": 385, ""y"": 46, ""width"": 139, ""height"": 20}",NaN,NaN,NaN,row:4 key:annot_cue_005,extraction_0q7ze135fhs95
5,annot_cue_006,NaN,interval,rectangle,71.0,75.75,4.75,thoresen_2010_form-building-patterns_p90-91_pa...,rect_d3,NaN,"{""x"": 310, ""y"": 93, ""width"": 154, ""height"": 18}",NaN,NaN,NaN,row:5 key:annot_cue_006,extraction_0q7ze135fhs95
6,annot_cue_007,NaN,interval,rectangle,76.0,83.50,7.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_b3,NaN,"{""x"": 456, ""y"": 69, ""width"": 229, ""height"": 18}",NaN,NaN,NaN,row:6 key:annot_cue_007,extraction_0q7ze135fhs95
7,annot_cue_008,NaN,interval,rectangle,90.5,94.50,4.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_i4,NaN,"{""x"": 14, ""y"": 115, ""width"": 127, ""height"": 31}",NaN,NaN,NaN,row:7 key:annot_cue_008,extraction_0q7ze135fhs95
8,annot_cue_009,NaN,interval,rectangle,113.4,116.40,3.00,thoresen_2010_form-building-patterns_p90-91_pa...,rect_a4,NaN,"{""x"": 663, ""y"": 82, ""width"": 97, ""height"": 23}",NaN,NaN,NaN,row:8 key:annot_cue_009,extraction_0q7ze135fhs95
9,annot_cue_010,NaN,interval,rectangle,121.0,128.50,7.50,thoresen_2010_form-building-patterns_p90-91_pa...,rect_i5,NaN,"{""x"": 19, ""y"": 119, ""width"": 251, ""height"": 29}",NaN,NaN,NaN,row:9 key:annot_cue_010,extraction_0q7ze135fhs95


In [11]:
# Graphical timeline (pixels)
graphical.events.to_dataframe()

,id,name,temporal_type,event_type,start,end,duration,event_id,alignment_group_id,start_time_sec,duration_sec,graphical_element_id,image_filename,text_content,text_anchor_xy_json,layer_order,description,locator,extraction_id
0,rectangle_18c9cr4wypp4e,NaN,interval,rectangle,10,158,148,annot_cue_001,NaN,0.0,5.00,rect_a,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:0,extraction_3u4qwlo24251l
1,rectangle_07z1ph5ttqgyo,NaN,interval,rectangle,40,167,127,annot_cue_002,NaN,1.5,4.00,rect_b,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:1,extraction_3u4qwlo24251l
2,rectangle_1xj3v1n67yzfj,NaN,interval,rectangle,111,168,57,annot_cue_003,NaN,3.5,2.00,rect_c,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:2,extraction_3u4qwlo24251l
3,rectangle_0bkh04yaszrl0,NaN,interval,rectangle,145,305,160,annot_cue_004,NaN,34.6,5.20,rect_a2,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:3,extraction_3u4qwlo24251l
4,rectangle_0c3r24w3zf650,NaN,interval,rectangle,385,524,139,annot_cue_005,NaN,43.5,4.50,rect_h2,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:4,extraction_3u4qwlo24251l
5,rectangle_1zcwervvdk71l,NaN,interval,rectangle,310,464,154,annot_cue_006,NaN,71.0,4.75,rect_d3,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:5,extraction_3u4qwlo24251l
6,rectangle_2x4ugb2xd0djj,NaN,interval,rectangle,456,685,229,annot_cue_007,NaN,76.0,7.50,rect_b3,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:6,extraction_3u4qwlo24251l
7,rectangle_15pvnl2341uvz,NaN,interval,rectangle,14,141,127,annot_cue_008,NaN,90.5,4.00,rect_i4,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:7,extraction_3u4qwlo24251l
8,rectangle_0dzkdx2l2lprb,NaN,interval,rectangle,663,760,97,annot_cue_009,NaN,113.4,3.00,rect_a4,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:8,extraction_3u4qwlo24251l
9,rectangle_272vb74bbuwkf,NaN,interval,rectangle,19,270,251,annot_cue_010,NaN,121.0,7.50,rect_i5,thoresen_2010_form-building-patterns_p90-91_pa...,NaN,NaN,NaN,NaN,row:9,extraction_3u4qwlo24251l


### Creating Timelines

Use `create_timeline()` to turn loaded events into a Timeline object:

In [12]:
# Physical timeline (seconds)
physical_tl = typed.create_timeline(uid="thoresen_physical")
physical_tl

ContinuousPhysicalTimeline(id='thoresen_physical', length=142.5, unit=seconds, events=11, children=0)

In [13]:
physical_tl.get_timestamp_table()

pyarrow.Table
thoresen_physical: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
thoresen_physical: [
  -- is_valid: all not null
  -- child 0 type: double
[0,1.5,3.5,5,5.5,...,116.4,121,128.5,141,142.5]
  -- child 1 type: int64
[0,3,7,5,11,...,4095460911140045,121,257,141,285]
  -- child 2 type: int64
[1,2,2,1,2,...,35184372088832,1,2,1,2]]

In [14]:
# Graphical timeline (pixels)
graphical_tl = graphical.create_timeline(uid="thoresen_graphical")
graphical_tl

DiscreteGraphicalTimeline(id='thoresen_graphical', length=760, unit=pixels, events=11, children=0)

**Note:** Both timelines represent the same 11 events in different
coordinate systems:
- **Physical:** `0 - 142.5 seconds` (audio time)
- **Graphical:** `10 - 760 pixels` (image coordinates)

TimeToAlign! uses these dual representations to align graphical
annotations with audio.

### Child Timelines from Column Values with `group_by`

When your data carries events from **multiple sources** (images, pages,
tracks), pass `group_by` to `create_timeline()` to split the events into
one child timeline per unique value. The Thoresen data has events from
five different image files:

In [15]:
from timetoalign.timelines import create_timeline  # noqa: E402

grouped_tl = create_timeline(typed, group_by="image_filename")
grouped_tl

ContinuousPhysicalTimeline(id='cpt_02ko73v5jgfso', length=142.5, unit=seconds, events=0, children=5)

Each child gets a minted ID (`cpt_…`) and is named after its
column value, so the image a child holds is its `name`:

In [16]:
# Each child timeline represents events from one image
{
    "parent_id": grouped_tl.id,
    "n_children": grouped_tl.n_children,
    "children": {
        child.id: (child.name, child.n_events)
        for _, child in grouped_tl.iter_children()
    },
}

{'parent_id': 'cpt_02ko73v5jgfso',
 'n_children': 5,
 'children': {'cpt_0wekdlw2l790w': ('thoresen_2010_form-building-patterns_p90-91_page1_1.jpeg',
   3),
  'cpt_0n6c88pget25z': ('thoresen_2010_form-building-patterns_p90-91_page1_2.jpeg',
   2),
  'cpt_2md3sif58dbaf': ('thoresen_2010_form-building-patterns_p90-91_page1_3.jpeg',
   2),
  'cpt_1zjspaw5ngu66': ('thoresen_2010_form-building-patterns_p90-91_page1_4.jpeg',
   2),
  'cpt_2tdv5kkt5jmef': ('thoresen_2010_form-building-patterns_p90-91_page2_1.jpeg',
   2)}}

***

## Summary

| Goal | How |
|------|-----|
| Load an ms3 score TSV | `Ms3Loader()` + `loader.load(path)` |
| Map your own columns | Subclass `TsvLoader` / `CsvLoader`, set `start_column` etc. |
| Promote a column to a typed field | Name it in `column_specs` (`{"col": int}`) |
| Reach a nested JSON value | `Field("column", "nested")` as a coordinate attribute |
| Derive a coordinate | `ComputedField("end", formula="...")` |
| Split into child timelines | `create_timeline(loader, group_by="column")` |

```python
from timetoalign.loader.tabular import TsvLoader

class MyLoader(TsvLoader):
    start_column = "onset"
    duration_column = "dur"
    column_specs = {"pitch": int, "velocity": int}
```

> **Key Takeaway:** Tabular loaders map CSV/TSV columns to TimeToAlign!
> events declaratively. Unnamed columns ride along as property columns;
> `column_specs` promotes the ones you want as typed fields; `Field`
> reaches nested JSON. For the full column-to-field mechanism — composite
> columns, semantic field types, and Step-2 promotion — continue to the
> in-depth CSV/TSV how-to.